# 13 - Ensemble Optimization & Stacking (No Leakage)

**Obiettivo:** Ottimizzare la combinazione dei modelli (XGBoost, Random Forest, MLP) per massimizzare l’F1-Score senza commettere Data Leakage.

**Metodologia Rigorosa:**
1.  **Calibrazione:** Usiamo il Training Set (o una sua partizione) per apprendere i parametri dell’ensemble (pesi, soglie).
2.  **Test:** Applichiamo i parametri appresi al Test Set **solo alla fine**.

**Strategie Implementate:**
1.  **Threshold Tuning:** Ottimizzazione della soglia di decisione per ogni singolo modello.
2.  **Weighted Soft Voting:** Ricerca matematica dei pesi ottimali ($w_1 \cdot P_{xgb} + w_2 \cdot P_{rf} + ...$) per massimizzare la metrica.
3.  **Stacking (Meta-Learner):** Utilizzo di una Logistic Regression che prende in input le probabilità dei modelli base e impara a correggere i loro errori.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import pickle
import json
import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_recall_curve, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize

# Configurazione Plot
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Device Setup per MLP
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device in uso: {device}")

Device in uso: mps


## 1. Caricamento Dati e Modelli

In [2]:
# Caricamento Dataset (V4 Hybrid - Clean)
print("Caricamento dati...")
X_train = pd.read_csv('../data/processed_v4_hybrid/X_train.csv')
y_train = pd.read_csv('../data/processed_v4_hybrid/y_train.csv')['BinaryIncidentGrade']
X_test = pd.read_csv('../data/processed_v4_hybrid/X_test.csv')
y_test = pd.read_csv('../data/processed_v4_hybrid/y_test.csv')['BinaryIncidentGrade']

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")

# Verifica Integrità Test Set
assert len(X_test) == 134671, f"ATTENZIONE: Il Test Set ha {len(X_test)} righe invece di 134,671! Controllare i dati."
print("✅ Integrità Test Set verificata.")

Caricamento dati...
Train shape: (200784, 43)
Test shape:  (134671, 43)
✅ Integrità Test Set verificata.


In [3]:
# Definizione Classe MLP (deve corrispondere a quella usata nel training)
class MLP(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.BatchNorm1d(32), nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)

models = {}

# 1. Carica XGBoost
try:
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('../models/xgboost_v2/model.json')
    models['XGBoost'] = xgb_model
    print("✅ XGBoost caricato")
except Exception as e: print(f"❌ Errore XGBoost: {e}")

# 2. Carica Random Forest
try:
    with open('../models/random_forest_v2/model.pkl', 'rb') as f:
        rf_model = pickle.load(f)
    models['RandomForest'] = rf_model
    print("✅ Random Forest caricato")
except Exception as e: print(f"❌ Errore Random Forest: {e}")

# 3. Carica MLP
try:
    mlp_model = MLP(X_train.shape[1]).to(device)
    mlp_model.load_state_dict(torch.load('../models/mlp_baseline/model_weights.pth', map_location=device))
    mlp_model.eval()
    models['MLP'] = mlp_model
    print("✅ MLP caricato")
except Exception as e: print(f"❌ Errore MLP: {e}")

✅ XGBoost caricato
✅ Random Forest caricato
✅ MLP caricato


## 2. Generazione Matrice delle Probabilità
Creiamo una matrice dove ogni colonna è la probabilità predetta da un modello.
Usiamo un **Validation Set** estratto dal Train per calibrare l’ensemble (per evitare di ottimizzare su dati già "imparati a memoria" o sul test set).

In [ ]:
# Split Train in Train/Calibration (80/20) per ottimizzare i pesi senza bias
X_train_sub, X_calib, y_train_sub, y_calib = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print(f"Set di Calibrazione: {X_calib.shape}")

# Funzione helper per ottenere probabilità
def get_probas(X_data, model_dict):
    probas = pd.DataFrame(index=X_data.index)
    
    # XGBoost
    if 'XGBoost' in model_dict:
        probas['XGBoost'] = model_dict['XGBoost'].predict_proba(X_data)[:, 1]
        
    # Random Forest
    if 'RandomForest' in model_dict:
        probas['RandomForest'] = model_dict['RandomForest'].predict_proba(X_data)[:, 1]
        
    # MLP
    if 'MLP' in model_dict:
        # Scala dati per MLP
        scaler = StandardScaler().fit(X_train) # Fit su tutto il train originale
        X_scaled = scaler.transform(X_data)
        X_tensor = torch.FloatTensor(X_scaled).to(device)
        with torch.no_grad():
            out = model_dict['MLP'](X_tensor)
            probas['MLP'] = torch.sigmoid(out).cpu().numpy().flatten()
            
    return probas

print("Generazione probabilità su Calibration Set...")
calib_probas = get_probas(X_calib, models)
print("Generazione probabilità su Test Set (per valutazione finale)...")
test_probas = get_probas(X_test, models)

print("Esempio probabilità (Calibration):")
print(calib_probas.head())

SyntaxError: unterminated string literal (detected at line 37) (206898191.py, line 37)

## Strategia 1: Threshold Tuning (Soglia Dinamica)
Il default è 0.5. Tuttavia, nei dataset sbilanciati, la soglia ottimale per massimizzare l’F1 è spesso diversa.
Troviamo la soglia migliore sul set di calibrazione.

In [ ]:
def find_best_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
    best_idx = np.nanargmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx]

best_thresholds = {}
print("Ottimizzazione Soglie su Calibration Set:")

for model_name in calib_probas.columns:
    thresh, best_f1 = find_best_threshold(y_calib, calib_probas[model_name])
    best_thresholds[model_name] = thresh
    print(f"  {model_name:<15}: Soglia Ottimale = {thresh:.4f} (Max F1 Calib = {best_f1:.4f})")

## Strategia 2: Optimized Weighted Voting
Cerchiamo i pesi $w$ tali che $\sum w_i = 1$ che massimizzano l’F1-Score (o la Log-Loss) dell’ensemble.

In [ ]:
# Funzione obiettivo da minimizzare (Negative F1)
def loss_func(weights, X_probs, y_true):
    # Normalizza pesi
    weights = np.array(weights)
    weights /= weights.sum()
    
    # Calcola probabilità pesata
    final_prob = np.sum(X_probs * weights, axis=1)
    
    # Calcola F1 con soglia 0.5 (o ottimizzata dinamica, qui usiamo 0.5 per stabilità gradiente)
    preds = (final_prob >= 0.5).astype(int)
    return -f1_score(y_true, preds)

model_names = list(calib_probas.columns)
n_models = len(model_names)
initial_weights = [1/n_models] * n_models # Pesi uguali all’inizio

bounds = [(0, 1)] * n_models
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

res = minimize(
    loss_func, 
    initial_weights, 
    args=(calib_probas.values, y_calib),
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = res.x / res.x.sum()
weights_dict = dict(zip(model_names, optimal_weights))

print("Pesi Ottimizzati (Weighted Voting):")
for model, w in weights_dict.items():
    print(f"  {model:<15}: {w:.4f}")

## Strategia 3: Stacking (Logistic Regression Meta-Learner)
Usiamo una Logistic Regression per combinare le probabilità. A differenza del voting lineare, la Logistic Regression può imparare relazioni non lineari (tramite la sigmoide) e pesare i modelli in base alla confidenza.

In [ ]:
print("Addestramento Meta-Learner (Logistic Regression) su Calibration Set...")

meta_model = LogisticRegression(random_state=42)
meta_model.fit(calib_probas, y_calib)

print(f"Coeff Meta-Learner: {dict(zip(model_names, meta_model.coef_[0]))}")
print(f"Intercept: {meta_model.intercept_[0]:.4f}")

## 3. Valutazione Finale sul Test Set
Ora applichiamo le strategie apprese sul **Test Set** (che non è stato usato per l’ottimizzazione).

In [ ]:
results = {}

# 1. Modello Singolo Migliore (con soglia ottimizzata)
best_single_model = max(models.keys(), key=lambda m: f1_score(y_calib, (calib_probas[m] >= best_thresholds[m]).astype(int)))
thresh = best_thresholds[best_single_model]
preds_single = (test_probas[best_single_model] >= thresh).astype(int)
results[f'Best Single ({best_single_model})'] = preds_single

# 2. Weighted Voting Ottimizzato
weighted_probs = np.sum(test_probas.values * optimal_weights, axis=1)
preds_weighted = (weighted_probs >= 0.5).astype(int) # Soglia 0.5 standard su ensemble pesato
results['Optimized Voting'] = preds_weighted

# 3. Stacking
stacking_probs = meta_model.predict_proba(test_probas)[:, 1]
preds_stacking = (stacking_probs >= 0.5).astype(int)
results['Stacking (LogReg)'] = preds_stacking

# Calcolo Metriche
metrics_df = []

print("=" *80)
print(f"{'STRATEGIA':<25} {'ACC':<10} {'PREC':<10} {'REC':<10} {'F1':<10} {'AUC':<10}")
print("="*80)

for name, preds in results.items():
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    # Per AUC usiamo probabilità diverse a seconda del metodo
    if 'Single' in name: prob = test_probas[best_single_model]
    elif 'Voting' in name: prob = weighted_probs
    else: prob = stacking_probs
    
    auc = roc_auc_score(y_test, prob)
    
    metrics_df.append({
        'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc
    })
    
    print(f"{name:<25} {acc:.4f}     {prec:.4f}     {rec:.4f}     {f1:.4f}     {auc:.4f}")

print("="*80)

# Trova il vincitore
metrics_df = pd.DataFrame(metrics_df)
best_strategy = metrics_df.loc[metrics_df['F1'].idxmax()]
print(f"
🏆 Strategia Vincente: {best_strategy['Model']} (F1: {best_strategy['F1']:.4f})")

## 4. Salvataggio Risultati
Salviamo le metriche e la configurazione dell’ensemble migliore.

In [ ]:
output_dir = '../models/ensemble_optimized'
os.makedirs(output_dir, exist_ok=True)

report = {
    "best_strategy": best_strategy['Model'],
    "metrics": metrics_df.to_dict(orient='records'),
    "parameters": {
        "thresholds": {k: float(v) for k, v in best_thresholds.items()},
        "weights": {k: float(v) for k, v in weights_dict.items()},
        "stacking_coef": {k: float(v) for k, v in zip(model_names, meta_model.coef_[0])}
    }
}

with open(f'{output_dir}/ensemble_report.json', 'w') as f:
    json.dump(report, f, indent=4)

print(f"Report salvato in {output_dir}/ensemble_report.json")

# Grafico Comparativo
plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x='Model', y='F1', palette='viridis')
plt.title('Confronto F1-Score Strategie Ensemble', fontsize=14)
plt.ylim(0.5, 1.0)
for i, v in enumerate(metrics_df['F1']):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/ensemble_comparison.png')
plt.show()